In [1]:
import Pkg

function find_flexopt_root(start_dir=joinpath(homedir(), "flexOPT"))
    candidates = String[]

    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end

    directory = abspath(expanduser(start_dir))

    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end

    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_file = joinpath(candidate, "src", "commonBatchs.jl")

        if isfile(project_file) && isfile(source_file)
            return candidate
        end
    end

    error("Cannot locate flexOPT. Set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)

@show VERSION Threads.nthreads() Base.active_project()

  Activating 

VERSION = v"1.12.6"
Threads.nthreads() = 1
Base.active_project() = "/home/ydenizhernandez/flexOPT/Project.toml"


project at `~/flexOPT`


"/home/ydenizhernandez/flexOPT/Project.toml"

In [2]:
Pkg.add("IJulia")
Pkg.add("PythonCall")
ENV["JULIA_PYTHONCALL_EXE"] = "/usr/bin/python3" #Where should the user look for this?
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"

using PythonCall

pybuiltins = pyimport("builtins")  # Access Python's built-in functions (e.g., str(), type()) to handle Python objects returned by cppyy.
pyimport("sys").executable

   Resolving package versions...
     Project No packages added to or removed from `~/flexOPT/Project.toml`
    Manifest No packages added to or removed from `~/flexOPT/Manifest.toml`
Precompiling packages...
   8819.3 ms  ✓ InlineStrings → ParsersExt
    666.4 ms  ✓ WeakRefStrings
  12477.3 ms  ✓ Conda
  12570.5 ms  ✓ ColorBrewer
  16939.9 ms  ✓ BenchmarkTools
  13298.1 ms  ✓ CSV
  37077.0 ms  ✓ SciMLBase → SciMLBasePythonCallExt
  57124.3 ms  ✓ Plots
   2543.4 ms  ✓ Plots → IJuliaExt
   2762.0 ms  ✓ Plots → FileIOExt
   2830.1 ms  ✓ Plots → GeometryBasicsExt
   2935.3 ms  ✓ Plots → UnitfulExt
   3970.4 ms  ✓ StatGeochem → StatGeochemPlotsExt
 105588.2 ms  ✓ Makie
   4873.9 ms  ✓ DomainSets → DomainSetsMakieExt
   5201.0 ms  ✓ SciMLBase → SciMLBaseMakieExt
  32805.8 ms  ✓ CairoMakie
  62376.4 ms  ✓ GLMakie
  18 dependencies successfully precompiled in 184 seconds. 687 already precompiled.
   Resolving package versions...
     Project No packages added to or removed from `~/flexOPT/Pro

Python: '/usr/bin/python3'

In [3]:
os = pyimport("os")
sys = pyimport("sys")

ROOT_LIB = "/home/ydenizhernandez/root-install/lib" #Should be the path to the root library, not the root source code. This is where the ROOT.so file is located.
PROJECT_ROOT = "/home/ydenizhernandez/ModularEarth" # Shoukd be the path to the root source code, where the CMakeLists.txt file is located.

# Match the configuration of your working Python notebook
sys.path.insert(0, ROOT_LIB)
os.chdir(PROJECT_ROOT)

# Attempt to import CERN ROOT
ROOT = pyimport("ROOT")

println("ROOT version: ", ROOT.gROOT.GetVersion())


ROOT version: 6.36.000


In [4]:
# ── Step 1: Load the compiled ModularEarth shared library ──────────────────
load_status = ROOT.gSystem.Load("./libModularEarth.so")
# 0 = loaded successfully, 1 = already loaded, negative = error
println("Library load status : ", pyconvert(Int, load_status),
        "  (0 = success, 1 = already loaded)")

ME = ROOT.ModularEarth
println("ModularEarth namespace bound : ", ME)

Library load status : 0  (0 = success, 1 = already loaded)
ModularEarth namespace bound : <namespace cppyy.gbl.ModularEarth at 0x35e563d0>


In [5]:
# ── Step 2: Construct Earth and load a PREM table ───────────────────────────
# prem_15layers is the smallest available table — builds quickly
earth = ME.Earth("JuliaTest", "prem_test", 60.0, 60.0)
println("Earth object created")

# Populate layersconfigs from the PREM table
earth.SetEarthModel("prem_test")

println("\n--- Model Summary ---")
earth.GetModelSummary()

Earth object created


Info in <TGeoManager::TGeoManager>: Geometry JuliaTest, JuliaTest Earth Model created
Info in <TGeoManager::SetTopVolume>: Top volume is EarthTopSphere. Master volume is EarthTopSphere
Info in <TGeoNavigator::BuildCache>: --- Maximum geometry depth set to 100



--- Model Summary ---

--- Layer 1 (i_ly = 1) ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer 2 (i_ly = 2) ---
  Sublayer 1 | rmin: 1221.5 | rmax: 3480 | ρ: 10.0861 | Z/A: 0.4691

--- Layer 3 (i_ly = 3) ---
  Sublayer 1 | rmin: 3480 | rmax: 5701 | ρ: 4.4872 | Z/A: 0.4954

--- Layer 4 (i_ly = 4) ---
  Sublayer 1 | rmin: 5701 | rmax: 5971 | ρ: 3.8983 | Z/A: 0.4954

--- Layer 5 (i_ly = 5) ---
  Sublayer 1 | rmin: 5971 | rmax: 6356 | ρ: 2.9 | Z/A: 0.4956

--- Layer 6 (i_ly = 6) ---
  Sublayer 1 | rmin: 6356 | rmax: 6371 | ρ: 1.02 | Z/A: 0.5525

--- Layer 7 (i_ly = 7) ---
  Sublayer 1 | rmin: 6371 | rmax: 6386 | ρ: 0.001 | Z/A: 0.4991


Python: None

## `SetSublayerUpdates` — Customising a sublayer before building

`SetSublayerUpdates` lets you overwrite the material properties of any individual sublayer in the model **after** loading a PREM table with `SetEarthModel` but **before** calling `BuiltEarth()`.  The change is applied directly to the internal `layersconfigs` vector, so the modified values are what ROOT sees when it constructs the geometry.

### Signature
```cpp
earth.SetSublayerUpdates(
    i_ly,               # Int  — 1-based index of the major geological layer
    i_sly,              # Int  — 1-based index of the sublayer within that layer
    rho_sly,            # Float64 — new bulk density  [g/cm³]
    eyield_sly,         # Float64 — new electron yield  Yₑ = Σ wᵢ·(Z/A)ᵢ
    UpdateMaterialName  # String  — label for the new ROOT TGeoMaterial
)
```

### Usage notes
| Rule | Detail |
|------|--------|
| **Must precede `BuiltEarth()`** | Calling it after the geometry is already built raises a warning and does nothing. |
| **Indices are 1-based** | Major layer 1 = Inner Core; check `GetModelSummary()` for the full index list. |
| **`atomicMass` is fixed to 1.0** | The implementation encodes the electron yield directly in the `atomicNumber` field and sets `atomicMass = 1`, matching the convention used by the rest of the library. |
| **Verify with `GetModelSummary()`** | Call it before *and* after to confirm the mutation was applied. |

### Quick example
```julia
# Override Inner Core (layer 0, sublayer 0) density and electron yield
earth.SetSublayerUpdates(1, 1, 10.0, 0.45, "MyCustomMaterial")
earth.GetModelSummary()   # confirm the change
earth.BuiltEarth()        # build geometry with updated values
```

In [6]:

earth.GetModelSummary()



--- Layer 1 (i_ly = 1) ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer 2 (i_ly = 2) ---
  Sublayer 1 | rmin: 1221.5 | rmax: 3480 | ρ: 10.0861 | Z/A: 0.4691

--- Layer 3 (i_ly = 3) ---
  Sublayer 1 | rmin: 3480 | rmax: 5701 | ρ: 4.4872 | Z/A: 0.4954

--- Layer 4 (i_ly = 4) ---
  Sublayer 1 | rmin: 5701 | rmax: 5971 | ρ: 3.8983 | Z/A: 0.4954

--- Layer 5 (i_ly = 5) ---
  Sublayer 1 | rmin: 5971 | rmax: 6356 | ρ: 2.9 | Z/A: 0.4956

--- Layer 6 (i_ly = 6) ---
  Sublayer 1 | rmin: 6356 | rmax: 6371 | ρ: 1.02 | Z/A: 0.5525

--- Layer 7 (i_ly = 7) ---
  Sublayer 1 | rmin: 6371 | rmax: 6386 | ρ: 0.001 | Z/A: 0.4991


Python: None

In [7]:

earth.BuiltEarth()
println("BuiltEarth() returned")

TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_1_1_1 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_1 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_2 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_3 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_4 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_5 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_6 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_7 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_8 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_9 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeW

BuiltEarth() returned


---
## `EarthPaths` — Neutrino Geometric Path Through the Earth

`EarthPaths` computes the sequence of ROOT geometry segments that a neutrino traverses
between an atmospheric entry point and a surface detector.  It reuses the
`TGeoManager` already built by `Earth::BuiltEarth()` — no second geometry is created.

| Method | Purpose |
|--------|---------|
| `EarthPaths(geoManager)` | Constructor — binds to an existing closed geometry |
| `SetDetectorLocation(lon, lat, R_det)` | Geographic position of the detector |
| `GetSegments(th, phi[, radians])` | Run the tracking loop; fill segment list |
| `PrintSegments()` | Print formatted table to stdout |
| `GetPathSegments()` | Access the `std::vector<PathSegment>` directly |
| `DisplayVectors()` | Draw geometry + path + markers on a ROOT canvas |

### Test 2 — `SetDetectorLocation`

Sets the detector position from geographic coordinates and builds the local
East / North / Up coordinate frame used by `GetSegments`.

```julia
paths.SetDetectorLocation(
    lon,    # longitude [degrees], East positive
    lat,    # latitude  [degrees], North positive
    R_det   # distance from Earth centre [km]
)
```

We place the detector at the **geographic North Pole** (lat = 90°) on the
Earth's surface (R = 6371 km).  This simplifies verification: the local
+z axis (radially outward) points along the global +Z axis, making the
expected Cartesian coordinates trivially `(0, 0, 6371)`.

In [8]:
# ── EarthPaths Test 2: SetDetectorLocation ───────────────────────────────────
det_lon = 0.0    # longitude  [deg]
det_lat = 90.0   # latitude   [deg]  — North Pole
det_R   = 6371.0 # radius     [km]   — Earth's mean surface radius

earth.SetDetectorLocation(det_lon, det_lat, det_R)
println("SetDetectorLocation called: lon=", det_lon, "° lat=", det_lat, "° R=", det_R, " km")

# Verify by checking that calling GetSegments later does not raise
# (a full assertion on r_det is done in Test 3 via path length)
println("\nTest 2 passed — SetDetectorLocation completed without error.")

SetDetectorLocation called: lon=0.0° lat=90.0° R=6371.0 km

Test 2 passed — SetDetectorLocation completed without error.


---
## `UpdateEarthPath` / `OriginalEarthPath` — Cache-based path re-evaluation

These two functions re-apply a material cache to the already-computed ROOT geometry,
so the expensive tracking step runs only once (inside `GetEarthPath`) and subsequent
material edits or baseline queries cost only a cache lookup.

| Function | Cache used | Use case |
|----------|-----------|----------|
| `UpdateEarthPath()` | `BlockMaterialCache` (live, editable) | Reflect any `SetBlock*` / `SetSublayer*` edits without re-tracking |
| `OriginalEarthPath()` | `BlockMaterialCacheOriginal` (frozen at `BuiltEarth()`) | Always returns the unmodified PREM propagation path |

**Call order requirement:** both functions require `GetEarthPath` to have been called first
to populate `MultiGeomPathCache`.  Calling them beforehand triggers a guard that returns
an empty vector and prints a diagnostic.

### Test 1 — Guard: calling before `GetEarthPath`

`MultiGeomPathCache` is empty here because `GetEarthPath` has not been called yet
in this section.  Both functions must return an empty `vector<PropaPath>` and print
a diagnostic to `stderr`.

In [9]:
# ── Test 1: Guard — before GetEarthPath ──────────────────────────────────────
# If a previous cell already populated MultiGeomPathCache this guard may not fire;
# the important thing is that neither function crashes on an empty cache.

println("─── UpdateEarthPath guard ───────────────────────────────────────────────")
result_guard = earth.UpdateEarthPath()
n_guard = pyconvert(Int, result_guard.size())
println("UpdateEarthPath returned $n_guard path(s)  (expected 0 when cache is empty)")

println("\n─── OriginalEarthPath guard ─────────────────────────────────────────────")
result_guard2 = earth.OriginalEarthPath()
n_guard2 = pyconvert(Int, result_guard2.size())
println("OriginalEarthPath returned $n_guard2 path(s)  (expected 0 when cache is empty)")

println("\n✓ Test 1 — guard check complete (see stderr for diagnostic messages).")


─── UpdateEarthPath guard ───────────────────────────────────────────────
UpdateEarthPath returned 0 path(s)  (expected 0 when cache is empty)

─── OriginalEarthPath guard ─────────────────────────────────────────────
OriginalEarthPath returned 0 path(s)  (expected 0 when cache is empty)

✓ Test 1 — guard check complete (see stderr for diagnostic messages).


[UpdateEarthPath] GeomCache is empty — call GetEarthPath first.
[OriginalEarthPath] GeomCache is empty — call GetEarthPath first.


### Test 2 — `GetEarthPath` setup + `OriginalEarthPath` returns PREM values

Call `GetEarthPath` with a 3-angle grid to populate `MultiGeomPathCache`, then
verify that `OriginalEarthPath` returns the unmodified PREM densities even before
any edits are applied.

PREM reference (prem_test table, Inner Core sublayer 1): ρ ≈ 12.9807 g/cm³, Ye ≈ 0.4691.

In [16]:
# ── Test 2: GetEarthPath baseline + OriginalEarthPath ────────────────────────
# Reset any prior cache edits so we start from a clean PREM state.
earth.ResetBlockMaterialCache()

# 3-angle grid: oblique (120°), steep (150°), vertical (180°).
th_vals = [180.0,120.0,150.0]

th_grid = ROOT.std.vector["double"]()

for th in th_vals;  th_grid.push_back(th);  end

phi = 0.0   # azimuth


0.0

In [17]:

# GetEarthPath runs ROOT tracking and stores geometry in MultiGeomPathCache.
paths_ref = earth.GetEarthPath(th_grid, phi)


[Earth::GetEarthPath] Building Earth paths for 3 angles.
[Earth::GetEarthPath] Free cache  paths for 3 angles.
Building PropaPaths for 0 angles.
Finished building PropaPath for 0 angles.
Building PropaPaths for 1 angles.
Finished building PropaPath for 1 angles.
Building PropaPaths for 2 angles.
Finished building PropaPath for 2 angles.
[Earth::GetEarthPath] Finished building paths for 3 angles.


Python: vector<ModularEarth::PropaPath>{ @0x3e14bbf0, @0x3e14bc60, @0x3e14bcd0 }

In [ ]:


N_paths   = pyconvert(Int, paths_ref.size())

println("─── GetEarthPath ($(length(th_vals)) angles) ────────────────────────────────────────")
@assert N_paths == length(th_vals) "FAIL: expected $(length(th_vals)) PropaPath objects, got $N_paths"

for i in 0:(N_paths-1)
    pp     = paths_ref[i]
    N_segs = pyconvert(Int, pp.density.size())
    println("  Path $(i+1):")
    for j in 0:(N_segs-1)
        baseline = pyconvert(Float64, pp.baseline[j])
        density  = pyconvert(Float64, pp.density[j])
        eyield   = pyconvert(Float64, pp.eyield[j])
        ne   = pyconvert(Float64, pp.ne[j])
        println("    Segment $(j+1): baseline=$(baseline) km, density=$(density) g/cm³, eyield=$(eyield), ne=$(ne)")
    end
end





─── GetEarthPath (3 angles) ────────────────────────────────────────
  Path 1:
    Segment 1: baseline=15.0 km, density=0.001 g/cm³, eyield=0.4991, ne=0.0004991
    Segment 2: baseline=14.999990000000253 km, density=1.02 g/cm³, eyield=0.5525, ne=0.56355
    Segment 3: baseline=384.99999000000025 km, density=2.9 g/cm³, eyield=0.4956, ne=1.4372399999999999
    Segment 4: baseline=269.99999000000025 km, density=3.8983 g/cm³, eyield=0.4954, ne=1.9312178199999999
    Segment 5: baseline=2220.9999900000003 km, density=4.4872 g/cm³, eyield=0.4954, ne=2.2229588799999997
    Segment 6: baseline=2258.49999 km, density=10.0861 g/cm³, eyield=0.4691, ne=4.7313895100000005
    Segment 7: baseline=2442.9999900000003 km, density=12.9807 g/cm³, eyield=0.4691, ne=6.089246370000001
    Segment 8: baseline=2258.49999 km, density=10.0861 g/cm³, eyield=0.4691, ne=4.7313895100000005
    Segment 9: baseline=2220.99999 km, density=4.4872 g/cm³, eyield=0.4954, ne=2.2229588799999997
    Segment 10: baseline=269.


 *** Break *** user-defined signal 2: printing stacktrace
 Generating stack trace...

 *** Break *** user-defined signal 2: printing stacktrace
 Generating stack trace...

 *** Break *** user-defined signal 2: printing stacktrace
 Generating stack trace...
 0x0000707a5563feaa in <unknown> from /home/ydenizhernandez/.julia/artifacts/a95373603fb869dcd86b9f7191e57a4918ba6d2c/lib/libzmq.so
 0x0000707a52a408f7 in <unknown> from /home/ydenizhernandez/.julia/compiled/v1.12/IJulia/nfu7T_SSZBC.so
 0x0000707a86d47339 in uv_run at /workspace/srcdir/libuv/src/unix/core.c:434 from /home/ydenizhernandez/.julia/juliaup/julia-1.12.6+0.x64.linux.gnu/bin/../lib/julia/libjulia-internal.so.1.12
 0x0000707a86cb39f4 in ijl_task_get_next at /cache/build/builder-amdci5-4/julialang/julia-release-1-dot-12/src/scheduler.c:457 from /home/ydenizhernandez/.julia/juliaup/julia-1.12.6+0.x64.linux.gnu/bin/../lib/julia/libjulia-internal.so.1.12
 0x0000707a73a5266f in <unknown> from /home/ydenizhernandez/.julia/juliaup

: 

In [ ]:


N_paths   = pyconvert(Int, paths_ref.size())

println("─── GetEarthPath ($(length(th_vals)) angles) ────────────────────────────────────────")
@assert N_paths == length(th_vals) "FAIL: expected $(length(th_vals)) PropaPath objects, got $N_paths"

for i in 0:(N_paths-1)
    pp     = paths_ref[i]
    N_segs = pyconvert(Int, pp.density.size())
    println("  Path $(i+1):")
    for j in 0:(N_segs-1)
        baseline = pyconvert(Float64, pp.baseline[j])
        density  = pyconvert(Float64, pp.density[j])
        eyield   = pyconvert(Float64, pp.eyield[j])
        ne   = pyconvert(Float64, pp.ne[j])
        println("    Segment $(j+1): baseline=$(baseline) km, density=$(density) g/cm³, eyield=$(eyield), ne=$(ne)")
    end
end



In [ ]:

# ── OriginalEarthPath ─────────────────────────────────────────────────────────
orig_paths = earth.OriginalEarthPath()
N_orig     = pyconvert(Int, orig_paths.size())

println("\n─── OriginalEarthPath ───────────────────────────────────────────────────")
@assert N_orig == N_paths "FAIL: OriginalEarthPath returned $N_orig paths, expected $N_paths"

# Spot-check: first segment of the vertical path (θ=180°) enters the inner core.
prem_rho_ic = 12.9807
prem_ye_ic  = 0.4691

pp_vert       = orig_paths[N_paths-1]          # 0-based; θ=180° is last
N_segs_vert   = pyconvert(Int, pp_vert.density.size())
rho0_orig     = pyconvert(Float64, pp_vert.density[0])
ye0_orig      = pyconvert(Float64, pp_vert.eyield[0])
ne0_orig      = pyconvert(Float64, pp_vert.ne[0])

println("  Vertical path (θ=180°): $N_segs_vert segments")
println("  Seg[0] density = $rho0_orig  (expected ≈ $prem_rho_ic)")
println("  Seg[0] eYield  = $(round(ye0_orig; digits=4))  (expected ≈ $prem_ye_ic)")
println("  Seg[0] ne      = $(round(ne0_orig; digits=4))  (expected ≈ $(round(prem_rho_ic*prem_ye_ic; digits=4)))")

@assert abs(rho0_orig - prem_rho_ic) < 1e-3  "FAIL: OriginalEarthPath density ($rho0_orig) ≠ PREM ($prem_rho_ic)"
@assert abs(ye0_orig  - prem_ye_ic)  < 1e-3  "FAIL: OriginalEarthPath eYield ($ye0_orig) ≠ PREM ($prem_ye_ic)"
@assert abs(ne0_orig  - prem_rho_ic * prem_ye_ic) < 1e-3  "FAIL: ne ($ne0_orig) ≠ ρ·Ye ($(prem_rho_ic*prem_ye_ic))"

println("\n✓ Test 2 passed — OriginalEarthPath returns PREM values for unmodified model.")


### Test 3 — `UpdateEarthPath` reflects material cache edits; `OriginalEarthPath` unchanged

After calling `SetBlockDensity` on a block that lies on the tracked path,
`UpdateEarthPath` must return the new density while `OriginalEarthPath`
continues to return the original PREM value.

In [ ]:
# ── Test 3: UpdateEarthPath reflects SetBlockDensity ─────────────────────────
# Modify inner core block [1,1,1] — always the first segment of the vertical path.
new_rho_ic = 99.0
new_ye_ic  = 0.30
earth.SetBlockDensity(1, 1, 1, new_rho_ic)
earth.SetBlockEYield(1, 1, 1, new_ye_ic)
println("─── Applied SetBlockDensity/EYield(1,1,1) → ρ=$new_rho_ic, Ye=$new_ye_ic ──────")

# UpdateEarthPath: must reflect the edit without re-running ROOT tracking.
upd_paths = earth.UpdateEarthPath()
N_upd     = pyconvert(Int, upd_paths.size())
@assert N_upd == N_paths "FAIL: UpdateEarthPath returned $N_upd paths, expected $N_paths"

pp_upd     = upd_paths[N_paths-1]
rho0_upd   = pyconvert(Float64, pp_upd.density[0])
ye0_upd    = pyconvert(Float64, pp_upd.eyield[0])
ne0_upd    = pyconvert(Float64, pp_upd.ne[0])

println("  UpdateEarthPath  Seg[0]: ρ=$rho0_upd  Ye=$ye0_upd  ne=$(round(ne0_upd;digits=4))")
@assert abs(rho0_upd - new_rho_ic) < 1e-12  "FAIL: UpdateEarthPath density ($rho0_upd) ≠ $new_rho_ic"
@assert abs(ye0_upd  - new_ye_ic)  < 1e-12  "FAIL: UpdateEarthPath eYield ($ye0_upd) ≠ $new_ye_ic"
@assert abs(ne0_upd  - new_rho_ic * new_ye_ic) < 1e-12  "FAIL: ne ($ne0_upd) ≠ ρ·Ye ($(new_rho_ic*new_ye_ic))"

# OriginalEarthPath: must still return unmodified PREM — the edit must not bleed through.
orig_paths2  = earth.OriginalEarthPath()
pp_orig2     = orig_paths2[N_paths-1]
rho0_orig2   = pyconvert(Float64, pp_orig2.density[0])
ye0_orig2    = pyconvert(Float64, pp_orig2.eyield[0])

println("  OriginalEarthPath Seg[0]: ρ=$rho0_orig2  Ye=$(round(ye0_orig2;digits=4))  (expected ≈ $prem_rho_ic / $prem_ye_ic)")
@assert abs(rho0_orig2 - prem_rho_ic) < 1e-3  "FAIL: OriginalEarthPath density changed after edit ($rho0_orig2 ≠ $prem_rho_ic)"
@assert abs(ye0_orig2  - prem_ye_ic)  < 1e-3  "FAIL: OriginalEarthPath eYield changed after edit ($ye0_orig2 ≠ $prem_ye_ic)"

println("\n✓ Test 3 passed — UpdateEarthPath sees the edit; OriginalEarthPath is unaffected.")


### Test 4 — `costh` / `azimuth` correctness and cross-call isolation

Each `PropaPath` returned by both functions must carry `costh = cos(θ)` and
`azimuth = φ` matching the grid passed to `GetEarthPath`.
Additionally, calling `OriginalEarthPath` must not alter the material state
visible to a subsequent `UpdateEarthPath` call.

In [ ]:
# ── Test 4: costh / azimuth + cross-call isolation ───────────────────────────
println("─── costh / azimuth consistency ─────────────────────────────────────────")
for i in 0:(N_paths-1)
    th_deg         = th_vals[i+1]
    expected_costh = cos(deg2rad(th_deg))

    costh_upd  = pyconvert(Float64, upd_paths[i].costh)
    costh_orig = pyconvert(Float64, orig_paths2[i].costh)
    az_upd     = pyconvert(Float64, upd_paths[i].azimuth)
    az_orig    = pyconvert(Float64, orig_paths2[i].azimuth)

    println("  θ=$(th_deg)°  expected costh=$(round(expected_costh; digits=5))")
    println("    UpdateEarthPath:   costh=$(round(costh_upd; digits=5))  azimuth=$az_upd")
    println("    OriginalEarthPath: costh=$(round(costh_orig; digits=5))  azimuth=$az_orig")

    @assert abs(costh_upd  - expected_costh) < 1e-10  "FAIL: UpdateEarthPath costh wrong for θ=$th_deg°"
    @assert abs(costh_orig - expected_costh) < 1e-10  "FAIL: OriginalEarthPath costh wrong for θ=$th_deg°"
    @assert abs(az_upd  - phi) < 1e-12  "FAIL: UpdateEarthPath azimuth wrong (got $az_upd)"
    @assert abs(az_orig - phi) < 1e-12  "FAIL: OriginalEarthPath azimuth wrong (got $az_orig)"
end
println("  ✓ costh and azimuth correct for all $(N_paths) paths in both functions.")

# Cross-call isolation: calling OriginalEarthPath must not reset the live cache edit.
println("\n─── Cross-call isolation ────────────────────────────────────────────────")
_ = earth.OriginalEarthPath()   # extra OriginalEarthPath call
upd_check = earth.UpdateEarthPath()
rho_check = pyconvert(Float64, upd_check[N_paths-1].density[0])
@assert abs(rho_check - new_rho_ic) < 1e-12  "FAIL: UpdateEarthPath density changed after calling OriginalEarthPath ($rho_check ≠ $new_rho_ic)"
println("  ✓ UpdateEarthPath still sees ρ=$rho_check after an OriginalEarthPath call.")

# Clean up for any further cells.
earth.ResetBlockMaterialCache()
println("  Cache reset to PREM baseline.")

println("\n✓ Test 4 passed — costh/azimuth correct; functions are fully independent.")


### `UpdateEarthPath` / `OriginalEarthPath` test summary

| # | Capability tested | Assertion |
|---|-------------------|-----------|
| 1 | Guard before `GetEarthPath` | Both return empty vector; diagnostic printed |
| 2 | `OriginalEarthPath` on clean model | First segment ρ and Ye match PREM; ne = ρ·Ye |
| 3 | `UpdateEarthPath` reflects `SetBlockDensity/EYield` | Modified values appear; `OriginalEarthPath` unchanged |
| 4 | `costh` / `azimuth` correctness | Match `cos(θ)` and `φ` passed to `GetEarthPath` |
| 4 | Cross-call isolation | `OriginalEarthPath` does not reset the live cache visible to `UpdateEarthPath` |